# Induction circuit analysis for a `replicate.py` run

ARENA 1.2-style analysis of one training run: its config, how the induction and previous-token scores
evolved, attention patterns on repeated random tokens, and the circuit (which heads matter, the
previous-token head's positional attention, OV copying).

How the model is set up for analysis (see `circuits.py`):
- The run's GPT-2 is converted to a TransformerLens `HookedTransformer` and **unrolled**: a looped run
  (1 layer × 2 loops) becomes 2 blocks holding copies of the same weights, so block 0 = loop 0 and
  block 1 = loop 1. A 2-layer control is just its 2 layers.
- LayerNorm is folded into the neighbouring weights, so each LayerNorm is only "divide by the
  vector's norm". The circuit math below sometimes ignores this per-token scale.

In [32]:
RUN = "rep_gpt2_L2x1_300M_s1"   # folder name inside checkpoints/
STEP = None                      # None = final checkpoint; or a step listed in phases.json

In [33]:
from pathlib import Path

import circuitsvis as cv
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import HTML, display
from transformer_lens import FactoredMatrix

from circuits import load_analysis
from replicate import make_ps_tokens

t.set_grad_enabled(False)
a = load_analysis(Path("checkpoints") / RUN, STEP)
model = a.tl
n_blocks, n_heads, d_head = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
tokens_per_step = a.args.batch_size * a.args.ctx
print(f"{RUN}: step {a.step} ({a.step * tokens_per_step / 1e6:.0f}M tokens); blocks: {a.labels}")

# colours: sequential blue for magnitudes, blue-grey-red for signed values, fixed categorical hues
SEQ = [[0, "#ffffff"], [0.05, "#cde2fb"], [0.2, "#86b6ef"], [0.4, "#3987e5"], [0.6, "#256abf"],
       [0.8, "#184f95"], [1, "#0d366b"]]
DIV = [[0, "#184f95"], [0.25, "#6da7ec"], [0.5, "#f0efec"], [0.75, "#e66767"], [1, "#9f2a29"]]
BLUE, ORANGE = "#2a78d6", "#eb6834"


def imshow(z, x=None, y=None, title="", diverging=False, zmin=None, zmax=None, labels=None, height=None, **kw):
    z = t.as_tensor(z).float()
    if diverging:
        lim = zmax if zmax is not None else z.abs().max().item()
        zmin, zmax = -lim, lim
    fig = px.imshow(z.numpy(), x=x, y=y, title=title, color_continuous_scale=DIV if diverging else SEQ,
                    zmin=zmin, zmax=zmax, labels=labels or {}, aspect="auto", **kw)
    fig.update_layout(template="plotly_white", height=height)
    fig.show()


head_names = [f"{b}.H{h}" for b in a.labels for h in range(n_heads)]

rep_gpt2_L2x1_300M_s1: step 18310 (300M tokens); blocks: ['layer0', 'layer1']


## 1. Training run config

In [34]:
args = vars(a.args)
looped = a.args.n_loops > 1
arch = (f"{a.args.n_layers} layer(s) x {a.args.n_loops} loops (weights shared across loops)" if looped
        else f"{a.args.n_layers} layers, no looping")
unique = {id(p): p for p in a.hf.parameters()}.values()
n_params = sum(p.numel() for p in unique)
n_block = sum(p.numel() for p in {id(p): p for blk in a.hf.transformer.h for p in blk.parameters()}.values())
print(f"architecture: {arch}")
print(f"parameters: {n_params / 1e6:.2f}M total, {n_block / 1e6:.2f}M in the transformer blocks (unique)")
print(f"heads: {n_heads} per block, size {d_head};  MLP width {model.cfg.d_mlp}")
display(pd.DataFrame({"value": {k: v for k, v in args.items()}}))
if a.phases:
    display(pd.DataFrame([{"checkpoint": k, "step": v, "tokens (M)": round(v * tokens_per_step / 1e6, 1),
                           "induction (best head)": next(r["ind_max"] for r in a.metrics if r["step"] >= v)}
                          for k, v in sorted(a.phases.items(), key=lambda kv: kv[1])]))

architecture: 2 layers, no looping
parameters: 53.56M total, 14.18M in the transformer blocks (unique)
heads: 8 per block, size 96;  MLP width 3072


,value
arch,gpt2
batch_size,16
bf16,True
ckpt_mtokens,"5,10,20,30,40,50,60,80,100"
ckpt_phases,"0.03,0.1,0.25"
ctx,1024
d_model,768
data_tokens,310000000
device,None
dropout,0.1


,checkpoint,step,tokens (M),induction (best head)
0,pre-formation,2800,45.9,0.025622
1,onset,3550,58.2,0.103332
2,mid-transition,4100,67.2,0.253476
3,end,18310,300.0,0.811678


## 2. Scores over training

Each score is the best head's: **induction** = attention from a token in the second copy of a repeated
random sequence to the token after its first occurrence; **previous-token** = attention to the
immediately preceding token. Dotted lines mark the saved checkpoints.

In [35]:
m = a.metrics
tok = [r["tokens"] / 1e6 for r in m]
fig = go.Figure()
for kind, key, colour in [("induction", "ind", BLUE), ("previous-token", "prev", ORANGE)]:
    for b, blk_label in enumerate(a.labels):
        fig.add_trace(go.Scatter(x=tok, y=[max(r[key][b]) for r in m], name=f"{kind}, best head in {blk_label}",
                                 line=dict(color=colour, width=2, dash="solid" if b == n_blocks - 1 else "dot")))
for label, s in a.phases.items():
    fig.add_vline(x=s * tokens_per_step / 1e6, line=dict(color="#999", width=1, dash="dot"),
                  annotation_text=label, annotation_textangle=-90, annotation_font_size=10)
fig.update_layout(title=f"{RUN}: best head's score over training", xaxis_title="training tokens (millions)",
                  yaxis_title="score (mean attention)", yaxis_range=[0, 1], hovermode="x unified",
                  template="plotly_white", height=450)
fig.show()

fig = px.line(x=tok, y=[r["val_loss"] for r in m], labels={"x": "training tokens (millions)", "y": "validation loss"},
              title="Validation loss")
fig.update_traces(line=dict(color=BLUE, width=2))
fig.update_layout(template="plotly_white", height=300)
fig.show()

In [36]:
# every head's score at the end of training (from the last evaluation logged during training)
last = m[-1]
imshow(t.tensor(last["ind"]), x=[f"H{h}" for h in range(n_heads)], y=a.labels, zmin=0, zmax=1,
       title=f"Induction score per head (step {last['step']})", height=250)
imshow(t.tensor(last["prev"]), x=[f"H{h}" for h in range(n_heads)], y=a.labels, zmin=0, zmax=1,
       title=f"Previous-token score per head (step {last['step']})", height=250)

## 3. This checkpoint: attention patterns on repeated random tokens, loss per token

25 random tokens, then the same 25 again (no BOS, as in training). An **induction head** shows a stripe
24 positions below the diagonal in the second half; a **previous-token head** shows the stripe just
below the diagonal.

In [37]:
HALF = 25
rep = make_ps_tokens(n=1, half=HALF, seed=1)
str_toks = [model.tokenizer.decode([i]) for i in rep[0].tolist()]
logits, cache = model.run_with_cache(rep)
for b, blk_label in enumerate(a.labels):
    display(HTML(f"<h3>{blk_label}</h3>"))
    display(cv.attention.attention_heads(attention=cache["pattern", b][0], tokens=str_toks,
                                         attention_head_names=[f"{blk_label}.H{h}" for h in range(n_heads)]))

In [38]:
# loss on each token: high in the first copy (random), low in the second copy if induction works
logprobs = logits.log_softmax(-1)[0, :-1].gather(-1, rep[0, 1:, None])[:, 0]
fig = px.line(x=list(range(1, 2 * HALF)), y=(-logprobs).tolist(), markers=True,
              labels={"x": "position of the predicted token", "y": "loss (nats)"},
              title="Per-token loss on the repeated sequence")
fig.add_vline(x=HALF + 0.5, line=dict(color="#999", dash="dot"), annotation_text="second copy starts")
fig.update_traces(line=dict(color=BLUE, width=2), marker=dict(size=8))
fig.update_layout(template="plotly_white", height=350)
fig.show()

## 4. The circuit

### 4.0 Which heads?
Scores on the standard test (100 sequences of 50 random tokens, repeated). By default the
previous-token head is the best one in block 0 and the induction head the best one in the last block;
override `PREV` / `IND` if you want to look at others.

In [39]:
batch = make_ps_tokens(n=100, half=50)
_, pat_cache = model.run_with_cache(batch, names_filter=lambda n: n.endswith("pattern"), return_type=None)
dest = t.arange(50, 100)
ind_scores = t.stack([pat_cache["pattern", b][:, :, dest, dest - 49].mean((0, 2)) for b in range(n_blocks)])
prev_scores = t.stack([pat_cache["pattern", b][:, :, dest, dest - 1].mean((0, 2)) for b in range(n_blocks)])
del pat_cache

PREV = (0, int(prev_scores[0].argmax()))
IND = (n_blocks - 1, int(ind_scores[-1].argmax()))
name = lambda bh: f"{a.labels[bh[0]]}.H{bh[1]}"
print(f"previous-token head: {name(PREV)} (prev-token score {prev_scores[PREV]:.2f})")
print(f"induction head:      {name(IND)} (induction score {ind_scores[IND]:.2f})")
if looped:
    print(f"(looped run: {name(PREV)} and {name(IND)} share weights with "
          f"{a.labels[1 - PREV[0]]}.H{PREV[1]} and {a.labels[1 - IND[0]]}.H{IND[1]})")

previous-token head: layer0.H7 (prev-token score 0.78)
induction head:      layer1.H2 (induction score 0.81)


### 4.1 Which heads matter? (ablation)
Zero out one head at a time (in one block only, so for a looped run "loop 0 H7" is removed in loop 0
but still runs in loop 1) and measure how much the loss on the second copy goes up.

In [40]:
abl = make_ps_tokens(n=20, half=50, seed=2)

def second_half_loss(fwd_hooks=()):
    # unembed only the positions that predict second-copy tokens (full logits would be ~400 MB)
    with model.hooks(fwd_hooks=list(fwd_hooks)):
        _, cc = model.run_with_cache(abl, names_filter="ln_final.hook_normalized", return_type=None)
    lg = cc["ln_final.hook_normalized"][:, 50:-1] @ model.W_U + model.b_U
    return t.nn.functional.cross_entropy(lg.flatten(0, 1), abl[:, 51:].flatten()).item()

base = second_half_loss()
increase = t.zeros(n_blocks, n_heads)
for b in range(n_blocks):
    for h in range(n_heads):
        def zero_head(z, hook, h=h):
            z[:, :, h] = 0
            return z
        increase[b, h] = second_half_loss([(f"blocks.{b}.attn.hook_z", zero_head)]) - base
print(f"second-copy loss with nothing ablated: {base:.2f} nats")
imshow(increase, x=[f"H{h}" for h in range(n_heads)], y=a.labels, diverging=True,
       title="Increase in second-copy loss when one head is zeroed (nats)", height=250)

second-copy loss with nothing ablated: 5.60 nats


### 4.2 Previous-token head: does it attend by position? (QK circuit on positional embeddings)
ARENA's previous-token head attends one position back using positional information only. Here:
the attention pattern the head would produce if its queries and keys saw only the positional
embeddings `W_pos` (normalised as LayerNorm would). A clean stripe one below the diagonal means
"previous token" is implemented positionally.

In [41]:
b, h = PREV
N = 40
pos = model.blocks[b].ln1(model.W_pos[None, :N])[0]          # LayerNorm's normalisation, as the head sees it
q = pos @ model.W_Q[b, h] + model.b_Q[b, h]
k = pos @ model.W_K[b, h] + model.b_K[b, h]
scores = (q @ k.T) / d_head ** 0.5
scores = scores.masked_fill(t.triu(t.ones(N, N, dtype=t.bool), 1), float("-inf"))
pos_pattern = scores.softmax(-1)
imshow(pos_pattern, labels={"x": "key position", "y": "query position"}, zmin=0, zmax=1,
       title=f"{name(PREV)}: attention from positional embeddings alone", height=450)
print(f"fraction of query positions whose top key is the previous position: "
      f"{(pos_pattern[1:].argmax(-1) == t.arange(N - 1)).float().mean():.2f}")

fraction of query positions whose top key is the previous position: 1.00


### 4.2b How much of the previous-token head's *actual* attention comes from position?
Here we split the real attention scores to see the relative contribution from token embedding and positional embedding. Block 0's input at position *p* is
`E_p + P_p` (token + positional embedding). Folded LayerNorm divides vector `v_p` in position *p*
by one number `s_p` (its norm). As we have fixed an input, `s_p` is a constant. The attention score is bilinear in the
two positions' vectors, `B(v_p, v_r) = (v_p / s_p) W_Q W_Kᵀ (v_r / s_r)ᵀ / √d_head`. We decompose it into 6 terms:

| term | formula | varies with |
|---|---|---|
| token × token | `B(E_p, E_r)` | query and key |
| token × position | `B(E_p, P_r)` | query and key |
| position × token | `B(P_p, E_r)` | query and key |
| position × position | `B(P_p, P_r)` | query and key |
| query bias × key token | `b_Q · E_r W_K / s_r` | key only |
| query bias × key position | `b_Q · P_r W_K / s_r` | key only |
| (query side) × key bias, `b_Q · b_K` | | query only: **softmax ignores these** |

We average the following over 100 repeated-random inputs, over the second half of the queries:
- **excess at the previous position**: how much each term raises the score at key *p − 1* above the average
  over all keys *r ≤ p* that the query sees (what softmax responds to);
- **share of the across-key variation**: each term's covariance with the total (centred across keys),
  divided by the total's variance.

Then we show the attention patterns you would get from the real scores, from the position × position terms only, and
from everything *except* those, all with the actual `s_p`.

In [42]:
assert PREV[0] == 0, "this split assumes the previous-token head is in block 0 (input = token + position)"
b, h = PREV
T = batch.shape[1]
_, c0 = model.run_with_cache(batch, names_filter=lambda n: n in ("blocks.0.ln1.hook_scale", "blocks.0.attn.hook_attn_scores"),
                             stop_at_layer=1)
s = c0["blocks.0.ln1.hook_scale"]                                   # [batch, pos, 1]: LayerNorm's actual divisor
E = model.W_E[batch] / s                                            # token embedding part, as the head sees it
P = model.W_pos[:T] / s                                             # positional part (broadcast over the batch)
WQ, WK, bQ, bK = model.W_Q[b, h], model.W_K[b, h], model.b_Q[b, h], model.b_K[b, h]
bil = lambda q, k: q @ k.transpose(-1, -2) / d_head ** 0.5         # [batch, query, key]
terms = {
    "token x token": bil(E @ WQ, E @ WK),
    "token x position": bil(E @ WQ, P @ WK),
    "position x token": bil(P @ WQ, E @ WK),
    "position x position": bil(P @ WQ, P @ WK),
    "query bias x key token": bil(bQ.expand(T, d_head), E @ WK),
    "query bias x key position": bil(bQ.expand(T, d_head), P @ WK),
}
query_only = ((((E + P) @ WQ) @ bK)[..., None] + bQ @ bK) / d_head ** 0.5   # constant across keys
valid = t.tril(t.ones(T, T, dtype=t.bool))
real = c0["blocks.0.attn.hook_attn_scores"][:, h]
err = (sum(terms.values()) + query_only - real).masked_fill(~valid, 0).abs().max().item()
print(f"terms reproduce the head's actual scores (max error {err:.1e})")

D = t.arange(50, T)                                                 # queries: the second half, as in the score
def centred(x):                                                     # subtract each query's mean over its keys
    x = x.masked_fill(~valid, 0)
    return (x - x.sum(-1, keepdim=True) / valid.sum(-1, keepdim=True)).masked_fill(~valid, 0)
total_c = centred(sum(terms.values()))
var = (total_c[:, D] ** 2).sum() / valid[D].sum() / len(batch)
rows = []
for term, x in terms.items():
    xc = centred(x)
    rows.append({"term": term,
                 "excess at previous position": xc[:, D, D - 1].mean().item(),
                 "share of across-key variation": ((xc[:, D] * total_c[:, D]).sum() / valid[D].sum() / len(batch) / var).item()})
table = pd.DataFrame(rows).set_index("term")
table.loc["total"] = table.sum()
display(table.round(3))
print(f"(real excess at the previous position: {centred(real)[:, D, D - 1].mean():.3f})")

terms reproduce the head's actual scores (max error 9.1e-06)


,excess at previous position,share of across-key variation
term,,
token x token,0.016,0.048
token x position,-0.210,0.093
position x token,0.001,0.064
position x position,6.793,0.749
query bias x key token,0.014,0.036
query bias x key position,-0.001,0.010
total,6.613,1.000


(real excess at the previous position: 6.613)


In [20]:
pos_only = terms["position x position"]
patterns = {"real scores": sum(terms.values()),
            "position x position only": pos_only,
            "everything except position x position": sum(terms.values()) - pos_only}
for label, sc_ in patterns.items():
    pat = sc_.masked_fill(~valid, float("-inf")).softmax(-1)
    print(f"{label:40s}: mean attention to the previous token {pat[:, D, D - 1].mean():.3f}")
    imshow(pat[0, :40, :40], labels={"x": "key position", "y": "query position"}, zmin=0, zmax=1,
           title=f"{name(PREV)}, first 40 positions: pattern from {label}", height=420)

real scores                             : mean attention to the previous token 0.610


position x position only                : mean attention to the previous token 0.683


everything except position x position   : mean attention to the previous token 0.010


### 4.3 Induction head: does it copy? (OV circuit)
In the second pass (block 1), the residual stream at a position is the sum of: the **token embedding**
`E` and **position embedding** `P` (the "direct stream"), **block 0's attention output**, and **block 0's
MLP output** (computed from the stream after attention). `E` depends only on the token and `P` only on
the position; the attention output, and hence the MLP output, depend on the other tokens in the sequence.
(LayerNorm is ignored in this discussion; see note below.)

**Effect via the token embedding.** The head's effect via `E` alone is `W_E W_OV W_U`, which is the map
"token attended to" → "contribution to the logits". The simplest copying mechanism would be if 
- this matrix were close to diagonal, so the logit it raises most is the token attended to
- this part of the output dominates the head's output from the other components. 

Below, we see how close the matrix is to diagonal. Top-k accuracy: how often a token's own logit is among
the top k of its row, over 2,000 random tokens (chance ≈ k / 50,257).

In [43]:
g = t.Generator().manual_seed(0)
sample = t.randperm(model.cfg.d_vocab, generator=g)[:2000]
b, h = IND
W_OV = model.W_V[b, h] @ model.W_O[b, h]
hits = {1: 0, 5: 0}
for i in range(0, len(sample), 250):                             # rows in chunks: 250 x 50,257 at a time
    rows = sample[i:i + 250]
    top5 = (model.W_E[rows] @ W_OV @ model.W_U).topk(5, dim=-1).indices
    hits[1] += (top5[:, 0] == rows).sum().item(); hits[5] += (top5 == rows[:, None]).any(-1).sum().item()
for k in (1, 5):
    print(f"{name(IND)}, W_E W_OV W_U: top-{k} accuracy {hits[k] / len(sample):.3f}")
sub = sample[:50]
imshow(model.W_E[sub] @ W_OV @ model.W_U[:, sub], diverging=True, labels={"x": "logit of token", "y": "token attended to"},
       title=f"{name(IND)}: W_E W_OV W_U on 50 random tokens", height=500)

layer1.H2, W_E W_OV W_U: top-1 accuracy 0.005
layer1.H2, W_E W_OV W_U: top-5 accuracy 0.022


**Real activations: what would the head do if it attended only to position *p*?** Take the input vector to the induction head at a position *p*, apply `W_OV W_U`, and
check the largest logit. If this is the token at *p*, and the largest logit dominates the rest, this indicates that if the head attends only to *p*, the head would copy *p*'s own token. 

Over every position *p* of the random-repeated sequence, we check:
- What percentage of the time is the token at *p* the top logit? In the top-5?
- How much larger is the logit for *p*'s own token than the average logit? 

The same check is also done for each component of the residual stream separately (token embedding, position
embedding, the previous-token head, block 0's other heads and bias, block 0's MLP). We also check how much of the head's output comes from each component.

LayerNorm does not affect this test: before `W_OV` it multiplies each position's vector by one positive
number (the same for all of its components), and the final LayerNorm is also one positive number per
position; neither changes which logit is largest, or the shares.

In [44]:
bi, hi = IND
bp, hp = PREV
assert bp < bi, "the previous-token head must be in an earlier block than the induction head"
t1 = batch[:50]
keep = {"hook_embed", "hook_pos_embed", f"blocks.{bi}.hook_resid_pre"} | \
       {f"blocks.{b}.{x}" for b in range(bi) for x in ("hook_attn_out", "attn.hook_z", "hook_mlp_out")}
_, c = model.run_with_cache(t1, names_filter=lambda n: n in keep, stop_at_layer=bi + 1)
# one head's output is its z vectors times its W_O (TransformerLens's use_attn_result would build a
# batch x pos x head x d_head x d_model temporary, gigabytes even for small batches)
head_out = lambda b, h: c[f"blocks.{b}.attn.hook_z"][:, :, h] @ model.W_O[b, h]
comps = {"token embedding E": c["hook_embed"], "position embedding P": c["hook_pos_embed"].expand_as(c["hook_embed"])}
for b in range(bi):
    attn = c[f"blocks.{b}.hook_attn_out"]
    if b == bp:
        comps[f"{name(PREV)} (previous-token head)"] = head_out(b, hp)
        comps[f"{a.labels[b]} other heads + bias"] = attn - head_out(b, hp)
    else:
        comps[f"{a.labels[b]} attention"] = attn
    comps[f"{a.labels[b]} MLP"] = c[f"blocks.{b}.hook_mlp_out"]
resid = c[f"blocks.{bi}.hook_resid_pre"]
print(f"components sum to the residual stream (max error {(sum(comps.values()) - resid).abs().max():.1e})")
comps = {"full residual stream": resid, **comps}

own = t1                                                          # the token at each position p
W_OVU = model.W_V[bi, hi] @ model.W_O[bi, hi] @ model.W_U          # [d_model, d_vocab], computed once
stats = {k: {"top1": 0., "top5": 0., "ranks": [], "excess": 0.} for k in comps}
for i in range(0, len(t1), 5):                                    # chunks keep memory down
    for k, x in comps.items():
        lg = x[i:i + 5] @ W_OVU                                   # [chunk, pos, d_vocab]
        own_lg = lg.gather(-1, own[i:i + 5, :, None])
        rank = (lg > own_lg).sum(-1)                              # 0 = own token has the largest logit
        stats[k]["top1"] += (rank == 0).float().sum().item()
        stats[k]["top5"] += (rank < 5).float().sum().item()
        stats[k]["ranks"].append(rank.flatten())
        stats[k]["excess"] += (own_lg[..., 0] - lg.mean(-1)).sum().item()
n = t1.numel()
total_excess = stats["full residual stream"]["excess"]
table = pd.DataFrame({k: {"top-1: own token wins": v["top1"] / n, "top-5": v["top5"] / n,
                          "median rank of own token": t.cat(v["ranks"]).float().median().item(),
                          "own-token logit above mean": v["excess"] / n,
                          "share of the total": v["excess"] / total_excess if k != "full residual stream" else 1.0}
                      for k, v in stats.items()}).T
display(table.round(3))
print(f"chance top-1: {1 / model.cfg.d_vocab:.1e}; the shares of the components sum to "
      f"{table['share of the total'].iloc[1:].sum():.3f}")
del c, comps, lg, W_OVU

components sum to the residual stream (max error 9.5e-07)


,top-1: own token wins,top-5,median rank of own token,own-token logit above mean,share of the total
full residual stream,0.948,0.968,0.0,14.346,1.000
token embedding E,0.004,0.015,3313.0,0.079,0.005
position embedding P,0.000,0.000,24736.0,-0.000,-0.000
layer0.H7 (previous-token head),0.000,0.000,25082.0,0.001,0.000
layer0 other heads + bias,0.000,0.001,16717.0,0.023,0.002
layer0 MLP,0.947,0.967,0.0,14.245,0.993


chance top-1: 2.0e-05; the shares of the components sum to 1.000


As we see, *p*'s own token is usually in the top 5 logits, and the contribution is mostly from the MLP.

In [45]:
seq = t1[0]
_, c1 = model.run_with_cache(seq[None], names_filter=f"blocks.{bi}.hook_resid_pre", stop_at_layer=bi + 1)
row_logits = c1[f"blocks.{bi}.hook_resid_pre"][0, :50] @ model.W_V[bi, hi] @ model.W_O[bi, hi] @ model.W_U
grid = row_logits[:, seq[:50]] - row_logits.mean(-1, keepdim=True)
tok_labels = [f"{p}: {model.tokenizer.decode([i])!r}" for p, i in enumerate(seq[:50].tolist())]
print(f"own token has the largest of these 50 logits in {(grid.argmax(-1) == t.arange(50)).float().mean():.0%} of rows")
imshow(grid, x=tok_labels, y=tok_labels, diverging=True,
       labels={"x": "logit of token", "y": "token at position p"},
       title=f"{name(IND)}: logits it would add if it attended only to position p (one sequence)", height=750)
del c1, row_logits

own token has the largest of these 50 logits in 100% of rows


### 4.4 Where does the induction head's attention score come from? (decomposing queries and keys)
The attention plots show that, for the repeated-random sequence, at position half+*p*, the induction head attends strongly to the token in position *p*+1. We would like to understand what part of the input contributes most to this attention score.

A standard test is to linearly decompose the input into its parts
(embedding, positional embedding, each earlier head, earlier MLPs, biases), and see how much each (query part, key part) pair contributes to the attention score, by pushing them through the
head's query and key maps. If the head works by K-composition through the residual stream the large entry is
**query part = embedding, key part = output of previous-token head**.

These models have an MLP between the two attention layers, so the previous-token head's output can
reach the induction head *through* it. To try to understand this effect, we further split block 0's MLP output in two:
MLP0= MLP(previous-token head zero-ablated) + MLP("change" caused by previous-token head)

Here, MLP(ablated) is computed and MLP(change) is defined to satisfy the equation above.
Large entries for **key part = MLP(change)** would support K-composition through the MLP.

The plot below shows each pair's contribution to *(attn score at the induction position) − (average score over
all keys that query sees)*. softmax ignores anything that raises every key equally, so this is the
part of the score that actually steers attention.

In [46]:
dec = make_ps_tokens(n=16, half=50, seed=3)
keep = {"hook_embed", "hook_pos_embed", f"blocks.{bi}.hook_resid_pre", f"blocks.{bi}.ln1.hook_scale",
        f"blocks.{bi}.attn.hook_attn_scores"} | {f"blocks.{b}.{x}" for b in range(bi) for x in ("attn.hook_z", "hook_mlp_out")}
_, c = model.run_with_cache(dec, names_filter=lambda n: n in keep, stop_at_layer=bi + 1)
bi, hi = IND
parts, names_ = [c["hook_embed"], c["hook_pos_embed"].expand_as(c["hook_embed"])], ["embed", "pos"]
for b in range(bi):
    for h in range(n_heads):
        parts.append(c[f"blocks.{b}.attn.hook_z"][:, :, h] @ model.W_O[b, h]); names_.append(f"{a.labels[b]}.H{h}")
    parts.append(model.b_O[b].expand_as(c["hook_embed"]).clone()); names_.append(f"{a.labels[b]} attn bias")
    if b == PREV[0]:
        # the MLP can pass the previous-token head's output on; split its output into what it would
        # be without that head, and the change the head causes (exact split, since they add up)
        def drop_prev(z, hook):
            z[:, :, PREV[1]] = 0
            return z
        with model.hooks(fwd_hooks=[(f"blocks.{b}.attn.hook_z", drop_prev)]):
            _, c_abl = model.run_with_cache(dec, names_filter=f"blocks.{b}.hook_mlp_out", stop_at_layer=b + 1)
        mlp_without = c_abl[f"blocks.{b}.hook_mlp_out"]
        parts += [mlp_without, c[f"blocks.{b}.hook_mlp_out"] - mlp_without]
        names_ += [f"{a.labels[b]} MLP (no {name(PREV)})", f"{a.labels[b]} MLP: change from {name(PREV)}"]
    else:
        parts.append(c[f"blocks.{b}.hook_mlp_out"]); names_.append(f"{a.labels[b]} MLP")
parts = t.stack(parts)                                                      # [part, batch, pos, d_model]
err = (parts.sum(0) - c[f"blocks.{bi}.hook_resid_pre"]).abs().max().item()
print(f"parts sum to the residual stream (max error {err:.1e})")

scale = c[f"blocks.{bi}.ln1.hook_scale"]                                    # LayerNorm's per-token divisor
qp = (parts / scale) @ model.W_Q[bi, hi]
kp = (parts / scale) @ model.W_K[bi, hi]
# the query and key biases are one more part each
qp = t.cat([qp, model.b_Q[bi, hi].expand_as(qp[:1])]); kp = t.cat([kp, model.b_K[bi, hi].expand_as(kp[:1])])
names_ = names_ + ["q/k bias"]
dest = t.arange(50, 100); src = dest - 49
# softmax only cares how much a key's score exceeds the other keys' scores for the same query, so
# measure each pair's contribution to (score at the induction position) - (mean score over keys 0..dest)
k_cummean = kp.cumsum(2) / t.arange(1, kp.shape[2] + 1)[:, None]             # mean key over positions 0..p
qs = qp[:, :, dest]                                                          # [part, batch, n_dest, d_head]
ks = kp[:, :, src] - k_cummean[:, :, dest]
contrib = t.einsum("ibnd,jbnd->ij", qs, ks) / (qs.shape[1] * qs.shape[2]) / d_head ** 0.5
scores_all = c[f"blocks.{bi}.attn.hook_attn_scores"][:, hi]                  # [batch, dest, src], masked = -inf
masked = scores_all.masked_fill(t.triu(t.ones(100, 100, dtype=t.bool), 1), 0)
excess = (scores_all[:, dest, src] - masked[:, dest].sum(-1) / (dest + 1)).mean().item()
print(f"induction-position score minus mean score over the query's keys: {excess:.3f}; "
      f"sum of all part pairs: {contrib.sum():.3f}")

top = contrib.flatten().abs().topk(6).indices
for i in top.tolist():
    qi, kj = divmod(i, len(names_))
    print(f"  query {names_[qi]:>16s}  x  key {names_[kj]:<16s}  {contrib[qi, kj]:+.2f}")
del c, parts, qp, kp, qs, ks
keep = contrib.abs().amax(0).topk(min(8, len(names_))).indices.sort().values   # the key parts that matter most
imshow(contrib[:, keep], x=[names_[i] for i in keep], y=names_, diverging=True,
       labels={"x": "key part", "y": "query part"},
       title=f"{name(IND)}: how much each (query part, key part) raises the induction position above the average key", height=550)

parts sum to the residual stream (max error 3.8e-06)
induction-position score minus mean score over the query's keys: 8.510; sum of all part pairs: 8.510
  query layer0 MLP (no layer0.H7)  x  key layer0 MLP: change from layer0.H7  +5.13
  query layer0 MLP (no layer0.H7)  x  key layer0 MLP (no layer0.H7)  +3.18
  query layer0 MLP: change from layer0.H7  x  key layer0 MLP: change from layer0.H7  -0.39
  query layer0 MLP (no layer0.H7)  x  key layer0.H7         +0.19
  query layer0 MLP: change from layer0.H7  x  key layer0 MLP (no layer0.H7)  +0.16
  query layer0 MLP (no layer0.H7)  x  key layer0.H2         +0.06


Below, we do a more refined test for K-composition through MLP0. We cut one route from the previous-token head to the induction head and measure the impact on attention to the correct token and on second-copy loss for the repeat-random sequence. 

By "cut one route", we mean: the output x from previous-token head reaches the induction head in 2 ways, either directly through the residual stream or through MLP0. For each of these, we replace x by x', the mean-ablation of the output. We use x' as an input for the induction head for one of its queries/keys/values at a time.

We see that changing x to x' and feeding it through the MLP to the keys has the most effect.

In [47]:
bi, hi = IND
bp, hp = PREV
assert bi == bp + 1, "written for a previous-token head in the block just before the induction head"
pp = make_ps_tokens(n=20, half=50, seed=4)
dest = t.arange(50, 100)

_, cl = model.run_with_cache(pp, stop_at_layer=bi + 1, names_filter=lambda n: n in
                             {f"blocks.{bp}.attn.hook_z", f"blocks.{bp}.hook_mlp_out", f"blocks.{bi}.hook_resid_pre"})
z = cl[f"blocks.{bp}.attn.hook_z"][:, :, hp]
z_mean = z.mean((0, 1))                                             # the head's average z vector

def mean_ablate(zz, hook):
    zz[:, :, hp] = z_mean
    return zz

with model.hooks(fwd_hooks=[(f"blocks.{bp}.attn.hook_z", mean_ablate)]):
    _, ab = model.run_with_cache(pp, names_filter=f"blocks.{bp}.hook_mlp_out", stop_at_layer=bp + 1)
x = cl[f"blocks.{bi}.hook_resid_pre"]
d_direct = (z_mean - z) @ model.W_O[bp, hp]                          # change in the head's own term
d_mlp = ab[f"blocks.{bp}.hook_mlp_out"] - cl[f"blocks.{bp}.hook_mlp_out"]   # change in the MLP's output
routes = {"direct": x + d_direct, "via MLP": x + d_mlp, "both": x + d_direct + d_mlp}
del cl, ab

QKV = {"keys": ("k", model.W_K, model.b_K), "queries": ("q", model.W_Q, model.b_Q), "values": ("v", model.W_V, model.b_V)}

def measure(hooks):
    # induction head's attention to the induction target, and the loss on the second copy
    with model.hooks(fwd_hooks=hooks):
        _, cc = model.run_with_cache(pp, return_type=None, names_filter=lambda n: n in
                                     {f"blocks.{bi}.attn.hook_pattern", "ln_final.hook_normalized"})
    attn = cc[f"blocks.{bi}.attn.hook_pattern"][:, hi][:, dest, dest - 49].mean().item()
    lg = cc["ln_final.hook_normalized"][:, 50:-1] @ model.W_U + model.b_U
    return attn, t.nn.functional.cross_entropy(lg.flatten(0, 1), pp[:, 51:].flatten()).item()

def patch_hook(route, use):
    hook_name, Wm, bm = QKV[use]
    new = model.blocks[bi].ln1(routes[route]) @ Wm[bi, hi] + bm[bi, hi]   # x' with its own LayerNorm scale
    def patch(act, hook):
        act[:, :, hi] = new
        return act
    return [(f"blocks.{bi}.attn.hook_{hook_name}", patch)]

rows = [("none", "-", *measure([]))]
rows.append((f"everywhere (ordinary ablation of {name(PREV)})", "all", *measure([(f"blocks.{bp}.attn.hook_z", mean_ablate)])))
for use in QKV:
    for route in routes:
        rows.append((route, use, *measure(patch_hook(route, use))))
pp_table = pd.DataFrame(rows, columns=["route cut", "used for", "induction attention", "second-copy loss"])
pp_table["change in attention"] = pp_table["induction attention"] - pp_table["induction attention"][0]
display(pp_table.round(3))
del routes, x, d_direct, d_mlp

,route cut,used for,induction attention,second-copy loss,change in attention
0,none,-,0.815,5.569,0.000
1,everywhere (ordinary ablation of layer0.H7),all,0.191,11.969,-0.624
2,direct,keys,0.803,5.635,-0.012
3,via MLP,keys,0.231,9.000,-0.583
4,both,keys,0.210,9.133,-0.605
5,direct,queries,0.814,5.574,-0.001
6,via MLP,queries,0.822,5.539,0.007
7,both,queries,0.820,5.549,0.006
8,direct,values,0.815,5.561,0.000
9,via MLP,values,0.815,5.266,0.000
